# GD Types: Batch, SGD, Mini-Batch + Momentum, RMSProp
---
> Simple English | Interview Ready

## Three Types of Gradient Descent

### 1. Batch GD
- Uses ALL training data for each update
- Very stable but very slow for large data

### 2. SGD (Stochastic)
- Uses 1 sample per update
- Fast but noisy (zigzag path)

### 3. Mini-Batch GD (Used in Practice)
- Batch of 32-256 samples per update
- Best balance of speed + stability
- Default in Keras/PyTorch

## Momentum — "Rolling Ball"
- Keeps running average of past gradients
- `v = b*v_prev + (1-b)*gradient`
- Faster convergence, escapes local minima

## Nesterov Momentum (Improvement)
- "Look ahead" before computing gradient
- Slightly more accurate than standard momentum
- `nesterov=True` in Keras SGD optimizer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
true_grad  = lambda w: 2*(w-3)
noisy_grad = lambda w: 2*(w-3) + np.random.randn()*2
lr=0.1; steps=30

# Batch GD
w=0.0; pb=[w]
for _ in range(steps): w-=lr*true_grad(w); pb.append(w)

# SGD
w=0.0; ps=[w]
for _ in range(steps): w-=lr*noisy_grad(w); ps.append(w)

# Mini-batch (average of 8 noisy gradients)
w=0.0; pm=[w]
for _ in range(steps):
    g=np.mean([noisy_grad(w) for _ in range(8)])
    w-=lr*g; pm.append(w)

plt.figure(figsize=(10,4))
plt.plot(range(steps+1),pb,'b-o',label='Batch GD (smooth)',lw=2)
plt.plot(range(steps+1),ps,'r-o',label='SGD (noisy)',lw=2,alpha=0.7)
plt.plot(range(steps+1),pm,'g-o',label='Mini-Batch (balanced)',lw=2)
plt.axhline(3,color='k',linestyle='--',label='Optimal w=3')
plt.legend(); plt.grid(True,alpha=0.3); plt.xlabel('Step'); plt.ylabel('w')
plt.title('Batch GD vs SGD vs Mini-Batch'); plt.show()

In [ ]:
# Nesterov Momentum (used in your Attrition project!)
import tensorflow as tf

# Exactly from your class code
optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.001,
    momentum=0.9,
    nesterov=True    # look-ahead momentum
)
print("Your Attrition project used:")
print("  SGD + Nesterov Momentum (lr=0.001, momentum=0.9)")
print()
print("Standard vs Nesterov Momentum:")
print("  Standard: compute gradient at w_t, then apply momentum")
print("  Nesterov: apply momentum first (peek ahead to w_t + momentum)")
print("            then compute gradient there")
print("  Result: Nesterov is more responsive to actual loss landscape")

In [ ]:
# RMSProp — adaptive learning rate
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
noisy_grad = lambda w: 2*(w-3)+np.random.randn()*2
lr=0.1; gamma=0.9; eps=1e-8; steps=30

# SGD
w=0.0; ps=[w]
for _ in range(steps): w-=lr*noisy_grad(w); ps.append(w)

# RMSProp
w=0.0; eg2=0; pr=[w]
for _ in range(steps):
    g=noisy_grad(w)
    eg2=gamma*eg2+(1-gamma)*g*g          # running avg of squared gradients
    w -= (lr/np.sqrt(eg2+eps))*g         # adaptive step
    pr.append(w)

plt.figure(figsize=(10,4))
plt.plot(range(steps+1),ps,'r-o',label='SGD',lw=2,alpha=0.7)
plt.plot(range(steps+1),pr,'b-o',label='RMSProp (adaptive LR)',lw=2)
plt.axhline(3,color='k',linestyle='--',label='Optimal w=3')
plt.legend(); plt.grid(True,alpha=0.3); plt.xlabel('Step'); plt.ylabel('w')
plt.title('SGD vs RMSProp'); plt.show()
print("RMSProp: large gradient -> smaller step | small gradient -> larger step")
print("Perfect for parameters with very different gradient scales (e.g. RNNs)")

## Interview Questions

**Q: Batch GD vs SGD vs Mini-batch?**
> Batch: stable, slow, needs all data in memory. SGD: one sample, fast updates, noisy. Mini-batch: subset (32-256), used in all modern deep learning — best of both.

**Q: What is Nesterov Momentum?**
> Look-ahead version of momentum. Compute where momentum would take you first, then compute gradient there. More responsive to actual gradient landscape. Slightly better than standard momentum.

**Q: Why use RMSProp over plain SGD?**
> RMSProp automatically adjusts learning rate per parameter based on recent gradient magnitudes. Parameters with large gradients get smaller steps; small gradients get larger steps. Essential for RNNs where gradient scales differ greatly.

**Q: What batch size should I use?**
> 32 or 64 is most common. Powers of 2 for GPU efficiency. Larger batch: faster training but sometimes worse generalization (sharp minima). Smaller batch: slower but often better test accuracy (flat minima).